In [7]:
import os
from langchain_community.document_loaders import TextLoader
from pathlib import Path
from langchain_community.document_loaders import DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:

def process_files(text_directory):
    all_documents = []
    text_dir = Path(text_directory)

    text_files = list(text_dir.glob("**/*.txt")) 
    print(f"Found {len(text_files)} text files in {text_directory}")

    for text_file in text_files:
        print(f"Processing file: {text_file.name}")
        loader = TextLoader(str(text_file))
        documents = loader.load()

        all_documents.extend(documents)
        print(f"  ✓ Loaded {len(documents)} pages")


    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

In [11]:
documents = process_files("docs/")

Found 7 text files in docs/
Processing file: bo.txt
  ✓ Loaded 1 pages
Processing file: himawari.txt
  ✓ Loaded 1 pages
Processing file: kazama.txt
  ✓ Loaded 1 pages
Processing file: masao.txt
  ✓ Loaded 1 pages
Processing file: misae.txt
  ✓ Loaded 1 pages
Processing file: nene.txt
  ✓ Loaded 1 pages
Processing file: shinchan.txt
  ✓ Loaded 1 pages

Total documents loaded: 7


In [12]:
documents

[Document(metadata={'source': 'docs\\bo.txt'}, page_content="Bo-chan is an exceptionally quiet, profoundly mysterious, and surprisingly brilliant five-year-old boy who moves at his own distinct pace. He speaks in incredibly slow, deliberate sentences, often remaining completely silent while his energetic friends scream and run around him. Despite his detached, blank expression, he possesses an immense depth of hidden knowledge, artistic talent, and deep philosophical wisdom.\n\nHis signature physical gag is a prominent, eternal stream of mucus that constantly hangs from his nose, which he controls with artistic precision. He can effortlessly manipulate this fluid, pulling it up or spinning it around to perform bizarre, gravity-defying tricks that amaze his classmates. This unusual physical trait functions as a natural extension of his unique, eccentric personality.\n\nHe is deeply obsessed with collecting unusually shaped rocks, unique pebbles, and various fascinating items found along

In [13]:
### chunking the documents into smaller pieces
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Total chunks created: {len(split_docs)}")

    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
        
    return split_docs

In [14]:
chunks = split_documents(documents)
chunks

Total chunks created: 32

Example chunk:
Content: Bo-chan is an exceptionally quiet, profoundly mysterious, and surprisingly brilliant five-year-old boy who moves at his own distinct pace. He speaks in incredibly slow, deliberate sentences, often rem...
Metadata: {'source': 'docs\\bo.txt'}


[Document(metadata={'source': 'docs\\bo.txt'}, page_content='Bo-chan is an exceptionally quiet, profoundly mysterious, and surprisingly brilliant five-year-old boy who moves at his own distinct pace. He speaks in incredibly slow, deliberate sentences, often remaining completely silent while his energetic friends scream and run around him. Despite his detached, blank expression, he possesses an immense depth of hidden knowledge, artistic talent, and deep philosophical wisdom.\n\nHis signature physical gag is a prominent, eternal stream of mucus that constantly hangs from his nose, which he controls with artistic precision. He can effortlessly manipulate this fluid, pulling it up or spinning it around to perform bizarre, gravity-defying tricks that amaze his classmates. This unusual physical trait functions as a natural extension of his unique, eccentric personality.'),
 Document(metadata={'source': 'docs\\bo.txt'}, page_content="He is deeply obsessed with collecting unusually shaped roc

In [16]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [17]:
class EmbeddingManaer:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):

        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        print(f"Loading embedding model: {self.model_name}...")
        self.model = SentenceTransformer(self.model_name)
        print(f"Loaded embedding model: {self.model_name}")    

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model is not loaded. Call _load_model() first.")
        print(f"Generating embeddings for {len(texts)} texts...")    
        embeddings = self.model.encode(texts, convert_to_numpy=True)
        print(f"Generated embeddings for {len(texts)} texts.")
        return embeddings    


In [18]:
embedding_manager = EmbeddingManaer()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 17985.57it/s]


Loaded embedding model: all-MiniLM-L6-v2
